# ResNet50 — Training & Evaluation (C1, C2, C3)

- **C1 (Replica):** Exact reproduction of reference (frozen base, simple head)
- **C2 (Improved):** Unfrozen base with better head and lower LR
- **C3 (Tuned):** Keras Tuner hyperparameter optimization

## 1. Setup

In [ ]:
# Standard library
import os
import random
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.regularizers import L2
import keras_tuner as kt

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

DATASET_PATH = Path(r"C:\Users\Owent\Desktop\ODL_assg\oral")

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 6
EPOCHS_BASELINE = 100      # Custom CNN improved configs
EPOCHS_PRETRAINED = 50     # Transfer learning improved configs
EPOCHS_TUNER = 30          # Keras Tuner trials
EPOCHS_REPLICA = 20        # Replica configs (matching reference)

# Set random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Create output directories
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

print("Configuration set.")

## 2. Data Pipeline

In [ ]:
def load_dataset(dataset_path: Path) -> pd.DataFrame:
    """Walk dataset directory, collect image paths and labels. Skips yolo folders."""
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset path does not exist: {dataset_path}")
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    records = []
    class_folders = [f for f in dataset_path.iterdir() if f.is_dir()]
    for class_folder in sorted(class_folders):
        class_name = class_folder.name
        if 'yolo' in class_name.lower():
            print(f"  [SKIPPED] {class_name}")
            continue
        class_count = 0
        for root, dirs, files in os.walk(class_folder):
            dirs[:] = [d for d in dirs if 'yolo' not in d.lower()]
            for fname in files:
                ext = os.path.splitext(fname)[1].lower()
                if ext in valid_extensions:
                    filepath = os.path.join(root, fname)
                    records.append({'filepath': filepath, 'label': class_name})
                    class_count += 1
        if class_count == 0:
            print(f"  [WARNING] No images in: {class_name}")
        else:
            print(f"  [OK] {class_name}: {class_count} images")
    if len(records) == 0:
        raise ValueError("No valid images found.")
    return pd.DataFrame(records)

# Load dataset
df = load_dataset(DATASET_PATH)
CLASS_NAMES = sorted(df['label'].unique().tolist())
LABEL_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
print(f"\nTotal: {len(df)} images, Classes: {CLASS_NAMES}")

# Stratified 70/15/15 split
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=47, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=47, stratify=temp_df['label'])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def process_path(filepath, label):
    """Read, decode, resize, normalize image; one-hot encode label."""
    img = tf.io.read_file(filepath)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    label = tf.one_hot(label, depth=NUM_CLASSES)
    return img, label

def build_dataset(dataframe, is_training=False):
    """Build batched, prefetched tf.data.Dataset from DataFrame."""
    filepaths = dataframe['filepath'].values
    labels = dataframe['label'].map(LABEL_TO_INDEX).values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    ds = ds.map(process_path, num_parallel_calls=AUTOTUNE)
    if is_training:
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = build_dataset(train_df, is_training=True)
val_ds = build_dataset(val_df, is_training=False)
test_ds = build_dataset(test_df, is_training=False)
print("Datasets built successfully.")

## 3. Utilities

In [ ]:
def get_augmentation_layers():
    """Augmentation layers embedded inside models (active only during training)."""
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
    ], name="augmentation")


def get_callbacks(checkpoint_path):
    """Fresh callback instances for each training run."""
    return [
        keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.1, patience=8),
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path, save_best_only=True, monitor="val_accuracy", mode="max"
        )
    ]


def save_history(history, filepath):
    """Save training history to CSV for later analysis."""
    hist = history if isinstance(history, dict) else history.history
    hist_df = pd.DataFrame(hist)
    hist_df.index.name = 'epoch'
    hist_df.to_csv(filepath)
    print(f"  History saved to: {filepath}")


def plot_training_history(history, model_name):
    """
    Plot training curves matching reference style:
    - Blue = Train, Red = Validation
    - Green dashed vertical line at best epoch
    - Green dot with annotation at best value
    - Text summary below
    """
    hist = history if isinstance(history, dict) else history.history
    
    epochs = range(1, len(hist['accuracy']) + 1)
    best_epoch = np.argmax(hist['val_accuracy'])
    best_val_acc = hist['val_accuracy'][best_epoch]
    best_val_loss = hist['val_loss'][best_epoch]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- Accuracy plot ---
    ax1.plot(epochs, hist['accuracy'], 'b-', linewidth=2, label='Train Accuracy')
    ax1.plot(epochs, hist['val_accuracy'], 'r-', linewidth=2, label='Validation Accuracy')
    ax1.axvline(x=best_epoch + 1, color='green', linestyle='--', linewidth=1.5, label=f'Best Epoch ({best_epoch + 1})')
    ax1.scatter(best_epoch + 1, best_val_acc, color='green', s=100, zorder=5)
    ax1.annotate(f'Acc: {best_val_acc:.4f}', xy=(best_epoch + 1, best_val_acc),
                 xytext=(best_epoch + 2, best_val_acc), fontsize=9, color='green')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.set_title(f'{model_name} Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # --- Loss plot ---
    ax2.plot(epochs, hist['loss'], 'b-', linewidth=2, label='Train Loss')
    ax2.plot(epochs, hist['val_loss'], 'r-', linewidth=2, label='Validation Loss')
    ax2.axvline(x=best_epoch + 1, color='green', linestyle='--', linewidth=1.5, label=f'Best Epoch ({best_epoch + 1})')
    ax2.scatter(best_epoch + 1, best_val_loss, color='green', s=100, zorder=5)
    ax2.annotate(f'Loss: {best_val_loss:.4f}', xy=(best_epoch + 1, best_val_loss),
                 xytext=(best_epoch + 2, best_val_loss), fontsize=9, color='green')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title(f'{model_name} Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary text
    print(f"  Best Epoch: {best_epoch + 1}")
    print(f"  Highest Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")
    print(f"  Lowest Validation Loss: {best_val_loss:.4f}")


def evaluate_model(model, train_ds, val_ds, test_ds, class_names, model_name):
    """Evaluate model on all splits, show classification report and confusion matrix."""
    train_loss, train_acc = model.evaluate(train_ds, verbose=0)
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    
    print(f"\n{'='*50}")
    print(f"{model_name} — Evaluation Results")
    print(f"{'='*50}")
    print(f"  Train — Acc: {train_acc:.4f}, Loss: {train_loss:.4f}")
    print(f"  Val   — Acc: {val_acc:.4f}, Loss: {val_loss:.4f}")
    print(f"  Test  — Acc: {test_acc:.4f}, Loss: {test_loss:.4f}")
    
    # Classification Report
    y_pred_probs = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.concatenate([np.argmax(labels.numpy(), axis=1) for _, labels in test_ds])
    
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{model_name} — Confusion Matrix')
    plt.tight_layout()
    plt.show()
    
    return {
        'model_name': model_name,
        'params_m': round(model.count_params() / 1e6, 2),
        'train_acc': train_acc,
        'val_acc': val_acc,
        'test_acc': test_acc,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'test_loss': test_loss,
    }

## 4. ResNet50 — C1 (Replica)

Exact replica of reference: ResNet50 frozen, GAP → Dense(128) → Dropout(0.5) → Dense(6). Adam(lr=0.001), 20 epochs.

In [ ]:
# --- ResNet50 C1: Exact replica ---

def build_resnet_c1():
    base = keras.applications.ResNet50(
        weights="imagenet", include_top=False, input_shape=(224, 224, 3)
    )
    base.trainable = False  # Frozen (same as reference)
    
    model = keras.Sequential([
        keras.Input(shape=(224, 224, 3)),
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ], name="ResNet50_C1_Replica")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy", metrics=["accuracy"]
    )
    return model

res_c1 = build_resnet_c1()
res_c1.summary()

history_res_c1 = res_c1.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_REPLICA,
    callbacks=get_callbacks("checkpoints/res_c1_best.h5")
)

save_history(history_res_c1, "logs/res_c1_history.csv")
plot_training_history(history_res_c1, "ResNet50 C1 (Replica)")

In [ ]:
# Evaluate ResNet50 C1
res_c1.load_weights("checkpoints/res_c1_best.h5")
result_res_c1 = evaluate_model(res_c1, train_ds, val_ds, test_ds, CLASS_NAMES, "ResNet50 C1")

## 5. ResNet50 — C2 (Improved)

Full fine-tuning (unfrozen) with better head: GAP → BN → Dense(256, L2) → Dropout(0.4) → Dense(6). Adam(lr=1e-4), 50 epochs.

In [ ]:
# --- ResNet50 C2: Improved (unfrozen) ---

def build_resnet_c2():
    inputs = keras.Input(shape=(224, 224, 3))
    x = get_augmentation_layers()(inputs)
    
    base = keras.applications.ResNet50(
        include_top=False, weights='imagenet', input_tensor=x
    )
    base.trainable = True  # Full fine-tuning
    
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=L2(0.0005))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name='ResNet50_C2_Improved')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

res_c2 = build_resnet_c2()
res_c2.summary()

history_res_c2 = res_c2.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_PRETRAINED,
    callbacks=get_callbacks("checkpoints/res_c2_best.h5")
)

save_history(history_res_c2, "logs/res_c2_history.csv")
plot_training_history(history_res_c2, "ResNet50 C2 (Improved)")

In [ ]:
# Evaluate ResNet50 C2
res_c2.load_weights("checkpoints/res_c2_best.h5")
result_res_c2 = evaluate_model(res_c2, train_ds, val_ds, test_ds, CLASS_NAMES, "ResNet50 C2")

## 6. ResNet50 — C3 (Keras Tuner)

BayesianOptimization with frozen base. Search: dense_units, dropout, l2_rate, learning_rate.

In [ ]:
# --- ResNet50 C3: Keras Tuner ---

def build_resnet_c3(hp):
    inputs = keras.Input(shape=(224, 224, 3))
    x = get_augmentation_layers()(inputs)
    
    base = keras.applications.ResNet50(
        include_top=False, weights='imagenet', input_tensor=x
    )
    base.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    
    dense_units = hp.Choice('dense_units', values=[128, 256, 512])
    dropout_rate = hp.Float('dropout_rate', min_value=0.2, max_value=0.6, step=0.1)
    l2_rate = hp.Choice('l2_rate', values=[1e-3, 5e-4, 1e-4])
    learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4, 5e-5])
    
    x = layers.Dense(dense_units, activation='relu', kernel_regularizer=L2(l2_rate))(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name='ResNet50_C3_Tuned')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

tuner_res = kt.BayesianOptimization(
    build_resnet_c3,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='resnet_c3',
    overwrite=True
)

print("Starting ResNet50 Keras Tuner search...")
tuner_res.search(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_TUNER,
    callbacks=get_callbacks("checkpoints/res_c3_search_best.h5")
)

best_hp_res = tuner_res.get_best_hyperparameters(1)[0]
print(f"\nBest ResNet50 hyperparameters:")
print(f"  dense_units: {best_hp_res.get('dense_units')}")
print(f"  dropout_rate: {best_hp_res.get('dropout_rate')}")
print(f"  l2_rate: {best_hp_res.get('l2_rate')}")
print(f"  learning_rate: {best_hp_res.get('learning_rate')}")

In [ ]:
# Retrain best ResNet50 C3
res_c3 = tuner_res.hypermodel.build(best_hp_res)
res_c3.summary()

history_res_c3 = res_c3.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_PRETRAINED,
    callbacks=get_callbacks("checkpoints/res_c3_best.h5")
)

save_history(history_res_c3, "logs/res_c3_history.csv")
plot_training_history(history_res_c3, "ResNet50 C3 (Tuned)")

res_c3.load_weights("checkpoints/res_c3_best.h5")
result_res_c3 = evaluate_model(res_c3, train_ds, val_ds, test_ds, CLASS_NAMES, "ResNet50 C3")

## 7. ResNet50 Family — Mini Comparison

In [ ]:
res_results = pd.DataFrame([result_res_c1, result_res_c2, result_res_c3])
res_results['Test Acc (%)'] = (res_results['test_acc'] * 100).round(2)
res_results['Val Acc (%)'] = (res_results['val_acc'] * 100).round(2)
print("\n=== ResNet50 Family Comparison ===")
display(res_results[['model_name', 'params_m', 'Val Acc (%)', 'Test Acc (%)', 'test_loss']].to_string(index=False))